In [ ]:
import os
import sys

# 1. MONTAGGIO DRIVE E SETUP PERCORSI
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory impostata su: {os.getcwd()}")
except ImportError:
    # Fallback per esecuzione in locale
    DRIVE_ROOT = '.'
    print("Esecuzione in locale")

# 2. INSTALLAZIONI DI SISTEMA (Usiamo ! bash commands per massima stabilità)
print("\n--- Inizio Installazione Detectron2 ---")
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

print("\n--- Allineamento Dipendenze (Numpy/iopath) ---")
!pip install --upgrade "iopath>=0.1.10" "fvcore>=0.1.5" "numpy<2.0"

print("\n--- Inizio Installazione SAM2 ---")
!rm -rf /content/sam2
!rm -rf /content/sam2_source
# Clonazione in /tmp, installazione statica senza "-e", e distruzione sorgente
!rm -rf /tmp/sam2_build
!git clone https://github.com/facebookresearch/sam2.git /tmp/sam2_build
!cd /tmp/sam2_build && pip install .
!rm -rf /tmp/sam2_build

print("\n--- Inizio Installazione VGGt ---")
!pip install git+https://github.com/facebookresearch/vggt.git

print("\n--- Inizio Installazione Transformers ---")
!pip install -U git+https://github.com/huggingface/transformers.git


!pip install "bitsandbytes>=0.46.1"

print("\n--- Utility Pipeline e Fix OpenCV ---")
!pip install supervision imageio scenedetect
!pip install "numpy<2.0"
!pip install "opencv-python==4.10.0.84" "opencv-contrib-python==4.10.0.84" "opencv-python-headless==4.10.0.84"


print("\n=======================================================")
print("INSTALLAZIONE COMPLETATA. RIAVVIO IL KERNEL AUTOMATICAMENTE.")
print("Attendi 5 secondi, poi esegui la cella con il tuo script Python.")
print("=======================================================")

# 3. RIAVVIO AUTOMATICO DEL RUNTIME
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory impostata su: {os.getcwd()}")
except ImportError:
    # Fallback per esecuzione in locale
    DRIVE_ROOT = '.'
    print("Esecuzione in locale")

# Cambia la directory di lavoro in modo che i path relativi (es. configs/, saved_models/) funzionino
os.chdir(DRIVE_ROOT)
print(f"Directory di lavoro ripristinata su: {os.getcwd()}")

In [ ]:
import sys
import os

video_dir='/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/VidOR/Videos_crop_decode'


if __name__ == '__main__':
  for robot in os.listdir(video_dir):
    print(robot)
    robot_dir = os.path.join(video_dir, robot)
    first_img ='0000000001.jpg'
    next_img = (int)(first_img.replace('.jpg', '')) + 1
    files = [f for f in os.listdir(robot_dir) if os.path.isfile(os.path.join(robot_dir, f))]
    if files == []:
      print("VUOTO: "+ robot)
    files.sort()
    for image in files:
      if ((int)(image.replace('.jpg', '')) +1) != next_img:
        print("Lacking: " + robot + " " + str(next_img))
      next_img+=1

In [ ]:
from pathlib import Path
import torch, random
import argparse
import os, sys, glob, imageio

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm
import json
from typing import Dict, Optional
import supervision as sv
from supervision.draw.color import Color, ColorPalette
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings("ignore")

from scipy.optimize import linear_sum_assignment

from utils_detectron2 import DefaultPredictor_Lazy

# sam2
from sam2.build_sam import build_sam2
from sam2.build_sam import build_sam2_video_predictor

from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import AutoModelForZeroShotObjectDetection

# VGGT for camera motion detection
from vggt.models.vggt import VGGT
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
import torch.nn.functional as F
VGGT_AVAILABLE = True
print("VGGT imported for camera motion detection")


if torch.cuda.is_available():
    torch.cuda.init()
    _ = torch.tensor([0.0], device="cuda")

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

def get_iou(bb1, bb2):
    if bb1[0] >= bb1[2] or bb1[1] >= bb1[3] or bb2[0] >= bb2[2] or bb2[1] >= bb2[3]:
        return 0.0

    x_left = max(bb1[0], bb2[0])
    y_top = max(bb1[1], bb2[1])
    x_right = min(bb1[2], bb2[2])
    y_bottom = min(bb1[3], bb2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)

    bb1_area = (bb1[2] - bb1[0]) * (bb1[3] - bb1[1])
    bb2_area = (bb2[2] - bb2[0]) * (bb2[3] - bb2[1])

    iou = intersection_area / float(bb1_area + bb2_area - intersection_area)
    return max(0.0, min(iou, 1.0))

def get_bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    return width * height

def chunk_into_n(lst, n):
    k, m = divmod(len(lst), n)
    chunks = []
    start = 0
    for i in range(n):
        end = start + k + (1 if i < m else 0)
        chunks.append(lst[start:end])
        start = end
    return chunks

def remove_contained_boxes(objects, containment_threshold=0.80):
    if not objects:
        return []

    objects_sorted = sorted(objects, key=lambda x: get_bbox_area(x['bbox']), reverse=True)
    kept_objects = []

    for obj in objects_sorted:
        bbox = obj['bbox']
        obj_area = max(get_bbox_area(bbox), 1e-6)
        is_contained = False

        for kept_obj in kept_objects:
            # SE LE CLASSI SONO DIVERSE, non applichiamo il filtro di contenimento.
            # Questo permette a una 'cake' o un 'phone' di esistere dentro la box 'person'.
            if kept_obj['class_name'] != obj['class_name']:
                continue

            kept_bbox = kept_obj['bbox']
            x_left = max(bbox[0], kept_bbox[0])
            y_top = max(bbox[1], kept_bbox[1])
            x_right = min(bbox[2], kept_bbox[2])
            y_bottom = min(bbox[3], kept_bbox[3])

            if x_right > x_left and y_bottom > y_top:
                intersection_area = (x_right - x_left) * (y_bottom - y_top)
                if (intersection_area / obj_area) > containment_threshold:
                    is_contained = True
                    break

        if not is_contained:
            kept_objects.append(obj)
    return kept_objects

def init_owlv2_model(device):
    model_id = "google/owlv2-large-patch14-ensemble"
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
    return model, processor

def detect_objects_owlv2(model, processor, image_array, queries, device, box_thresh=0.15):
    if len(image_array.shape) == 3 and image_array.shape[2] == 3:
        rgb_image = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
    else:
        rgb_image = image_array

    pil_image = Image.fromarray(rgb_image)
    texts = [queries]

    inputs = processor(text=texts, images=pil_image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([pil_image.size[::-1]]).to(device)

    results = processor.image_processor.post_process_object_detection(
        outputs=outputs,
        target_sizes=target_sizes,
        threshold=box_thresh
    )[0]

    final_boxes = []
    final_phrases = []
    final_scores = []

    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        final_boxes.append(box.tolist())
        final_scores.append(score.item())
        final_phrases.append(queries[label.item()])

    return final_boxes, final_phrases, final_scores

def filter_and_deduplicate_objects(objects, confidence_threshold=0.45, iou_threshold=0.3):
    if not objects:
        return []

    objects = sorted(objects, key=lambda x: x.get('confidence', 0), reverse=True)
    filtered_objects = []
    for obj in objects:
        if obj.get('confidence', 0) < confidence_threshold:
            continue

        is_duplicate = False
        for existing_obj in filtered_objects:
            iou = get_iou(obj['bbox'], existing_obj['bbox'])
            if iou > iou_threshold:
                if obj.get('confidence', 0) > existing_obj.get('confidence', 0):
                    filtered_objects.remove(existing_obj)
                else:
                    is_duplicate = True
                break

        if not is_duplicate:
            class_name = obj['class_name']
            name_duplicate = False
            for existing_obj in filtered_objects:
                if existing_obj['class_name'] == class_name:
                    bbox1 = obj['bbox']
                    bbox2 = existing_obj['bbox']
                    center1 = [(bbox1[0] + bbox1[2])/2, (bbox1[1] + bbox1[3])/2]
                    center2 = [(bbox2[0] + bbox2[2])/2, (bbox2[1] + bbox2[3])/2]
                    distance = ((center1[0] - center2[0])**2 + (center1[1] - center2[1])**2)**0.5

                    if distance < 50:
                        if obj.get('confidence', 0) > existing_obj.get('confidence', 0):
                            filtered_objects.remove(existing_obj)
                        else:
                            name_duplicate = True
                        break

            if not name_duplicate:
                filtered_objects.append(obj)

    return filtered_objects

# Inizializzazione dinamica delle classi e palette colori espansa (tab20)
OBJECTS_CLASS_ID = {
    'person': 0
}
cmap = plt.cm.get_cmap('tab20', 20)
HEX_COLOR_PALETTE = [mcolors.to_hex(cmap(i)) for i in range(20)]

import subprocess

def get_original_video_fps(video_name, video_base_dir):
    possible_paths = [
        os.path.join(video_base_dir, 'Videos_crop', f'{video_name}.mp4'),
    ]

    for video_path in possible_paths:
        if os.path.exists(video_path):
            try:
                cmd = [
                    "ffprobe", "-v", "error", "-select_streams", "v:0",
                    "-show_entries", "stream=r_frame_rate", "-of", "csv=p=0", video_path
                ]
                result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
                if result.returncode == 0:
                    fps_str = result.stdout.strip()
                    if '/' in fps_str:
                        num, den = fps_str.split('/')
                        fps = float(num) / float(den)
                    else:
                        fps = float(fps_str)
                    print(f"Found original video {video_path} with FPS: {fps}")
                    return fps
            except Exception as e:
                print(f"Failed to get FPS from {video_path}: {e}")
                continue
    return None

def calculate_fps_from_frames(frame_count, estimated_duration=5.0):
    return frame_count / estimated_duration

def init_vggt_model(device='cuda'):
    if not VGGT_AVAILABLE:
        return None
    model = VGGT()
    _URL = "https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt"
    model.load_state_dict(torch.hub.load_state_dict_from_url(_URL))
    model.eval()
    model = model.to(device)
    print(f"VGGT model initialized successfully on {device}")
    return model

def detect_camera_motion(vggt_model, image_paths, motion_threshold=0.1, sample_frames=8, device='cuda'):
    if vggt_model is None or len(image_paths) < 2:
        return False, 0.0, []

    total_frames = len(image_paths)
    if total_frames > sample_frames:
        indices = np.linspace(0, total_frames-1, sample_frames, dtype=int)
        sampled_paths = [image_paths[i] for i in indices]
    else:
        sampled_paths = image_paths

    torch.cuda.empty_cache()
    images_tensor_list = []
    for img_path in sampled_paths:
        image = Image.open(img_path).convert('RGB')
        image = image.resize((518, 518))
        image_array = np.array(image, dtype=np.float16)
        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1).float() / 255.0
        images_tensor_list.append(image_tensor)

    images_tensor = torch.stack(images_tensor_list).to(device)
    del images_tensor_list

    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            images_batch = images_tensor[None]
            aggregated_tokens_list, ps_idx = vggt_model.aggregator(images_batch)
            pose_enc = vggt_model.camera_head(aggregated_tokens_list)[-1]
            extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, images_tensor.shape[-2:])

        del images_tensor, images_batch, aggregated_tokens_list, pose_enc

    camera_poses = []
    extrinsic = extrinsic.squeeze(0).cpu().numpy()

    for i in range(len(extrinsic)):
        camera_matrix = extrinsic[i]
        translation = camera_matrix[:3, 3]
        rotation = camera_matrix[:3, :3]
        camera_poses.append({
            'translation': translation,
            'rotation': rotation,
            'matrix': camera_matrix
        })

    translation_changes = []
    rotation_changes = []

    for i in range(1, len(camera_poses)):
        pos_change = np.linalg.norm(camera_poses[i]['translation'] - camera_poses[i-1]['translation'])
        translation_changes.append(pos_change)
        rot_change = np.linalg.norm(camera_poses[i]['rotation'] - camera_poses[i-1]['rotation'], ord='fro')
        rotation_changes.append(rot_change)

    avg_translation_change = np.mean(translation_changes) if translation_changes else 0
    avg_rotation_change = np.mean(rotation_changes) if rotation_changes else 0
    max_translation_change = np.max(translation_changes) if translation_changes else 0
    max_rotation_change = np.max(rotation_changes) if rotation_changes else 0

    motion_score = (avg_translation_change + avg_rotation_change + max_translation_change * 0.5 + max_rotation_change * 0.5)
    is_moving = motion_score > motion_threshold
    print(f"Camera motion: score={motion_score:.4f}, threshold={motion_threshold}, is_moving={is_moving}")
    return is_moving, motion_score, camera_poses

def get_cached_image(image_path, cache, cache_limit=120):
    if image_path in cache:
        return cache[image_path]
    img = cv2.imread(str(image_path))
    if len(cache) >= cache_limit:
        oldest_key = next(iter(cache))
        del cache[oldest_key]
    cache[image_path] = img
    return img

def preload_video_images(img_paths, max_preload=50):
    preloaded = {}
    load_count = min(len(img_paths), max_preload)
    for i in range(load_count):
        img_path = img_paths[i]
        img = cv2.imread(str(img_path))
        if img is not None:
            preloaded[img_path] = img
    return preloaded

def should_skip_video_due_to_camera_motion(vggt_model, video_dir, motion_threshold=0.1, sample_frames=8):
    if vggt_model is None:
        return False, 0.0, "VGGT model not available"

    img_ls = glob.glob(f'{video_dir}/*.jpg')
    img_ls.sort()

    if len(img_ls) < 2:
        return False, 0.0, "Insufficient frames for motion detection"

    is_moving, motion_score, _ = detect_camera_motion(
        vggt_model, img_ls, motion_threshold=motion_threshold, sample_frames=sample_frames
    )

    if is_moving:
        reason = f"Camera is moving (score: {motion_score:.4f} > threshold: {motion_threshold})"
        return True, motion_score, reason
    else:
        reason = f"Camera is stable (score: {motion_score:.4f} <= threshold: {motion_threshold})"
        return False, motion_score, reason

def print_step_header(step_name, step_number=None):
    if step_number:
        print(f"Step {step_number}: {step_name}", flush=True)
    else:
        print(f"{step_name}", flush=True)

def check_and_merge_existing_markers(finished_dir, video_name):
    base_path = os.path.join(finished_dir, video_name)
    camera_motion_path = os.path.join(finished_dir, f'{video_name}_camera_motion_skipped')

    existing_markers = []
    if os.path.exists(base_path):
        existing_markers.append(('generic', base_path))
    if os.path.exists(camera_motion_path):
        existing_markers.append(('camera_motion', camera_motion_path))

    for item in os.listdir(finished_dir):
        if item.startswith(f'{video_name}_') and item != f'{video_name}_camera_motion_skipped':
            full_path = os.path.join(finished_dir, item)
            if os.path.isdir(full_path):
                existing_markers.append(('other', full_path))

    if len(existing_markers) <= 1:
        return base_path

    print(f"Found multiple failure markers for {video_name}: {[m[0] for m in existing_markers]}")

    if any(marker[0] == 'camera_motion' for marker in existing_markers):
        keep_path = camera_motion_path
        print(f"Keeping camera_motion_skipped marker (most informative)")
    else:
        keep_path = existing_markers[0][1]
        print(f"Keeping {existing_markers[0][0]} marker")

    for marker_type, marker_path in existing_markers:
        if marker_path != keep_path:
            try:
                if os.path.isdir(marker_path):
                    if os.listdir(marker_path):
                        print(f"Removing duplicate marker: {marker_path}")
                        import shutil
                        shutil.rmtree(marker_path)
                    else:
                        os.rmdir(marker_path)
                        print(f"Removed empty duplicate marker: {marker_path}")
            except Exception as e:
                print(f"Warning: Could not remove duplicate marker {marker_path}: {e}")

    return keep_path


if __name__ == '__main__':

    timing_stats = {
        'model_loading': 0,
        'camera_motion_detection': 0,
        'object_analysis_detection': 0,
        'sam2_propagation': 0,
        'reformatting_visualization': 0,
        'saving_results': 0,
        'total_video_processing': 0
    }

    script_start_time = time.time()

    video_dir='/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/VidOR'
    single_video_name=''
    video_names=[]
    chunk_num=10
    chunk_idx=-1
    body_detector='vitdet'
    enable_camera_motion_detection=False
    camera_motion_threshold=0.3
    camera_motion_sample_frames=4
    max_video_frames=300

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Single GPU mode: {device}")

    print("Loading models...", flush=True)
    model_loading_start = time.time()

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.cuda.empty_cache()

    # Load detector (Detectron2)
    if body_detector == 'vitdet':
        from detectron2.config import LazyConfig
        cfg_path = f'configs/cascade_mask_rcnn_vitdet_h_75ep.py'
        detectron2_cfg = LazyConfig.load(str(cfg_path))
        detectron2_cfg.train.init_checkpoint = "https://dl.fbaipublicfiles.com/detectron2/ViTDet/COCO/cascade_mask_rcnn_vitdet_h/f328730692/model_final_f05665.pkl"
        for i in range(3):
            detectron2_cfg.model.roi_heads.box_predictors[i].test_score_thresh = 0.25
        detector = DefaultPredictor_Lazy(detectron2_cfg)
    elif body_detector == 'regnety':
        from detectron2 import model_zoo
        from detectron2.config import get_cfg
        detectron2_cfg = model_zoo.get_config('new_baselines/mask_rcnn_regnety_4gf_dds_FPN_400ep_LSJ.py', trained=True)
        detectron2_cfg.model.roi_heads.box_predictor.test_score_thresh = 0.5
        detectron2_cfg.model.roi_heads.box_predictor.test_nms_thresh   = 0.4
        detector       = DefaultPredictor_Lazy(detectron2_cfg)

    # sam2
    sam2_checkpoint = "./saved_models/sam2_models/sam2.1_hiera_small.pt"
    model_cfg       = "configs/sam2.1/sam2.1_hiera_s.yaml"

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        torch.autocast(device_type='cuda', dtype=torch.float16).__enter__()

    predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint)
    predictor.to(dtype=torch.float16)

    owlv2_model, owlv2_processor = init_owlv2_model(device)

    vggt_model = None
    if enable_camera_motion_detection:
        print("Loading VGGT model for camera motion detection...")
        vggt_model = init_vggt_model(device)
        if vggt_model is not None:
            print("VGGT model loaded - camera motion detection enabled")
        else:
            print("Camera motion detection disabled")
    else:
        print("Camera motion detection disabled")

    model_loading_time = time.time() - model_loading_start
    timing_stats['model_loading'] = model_loading_time
    print(f"Models loaded in {model_loading_time:.1f}s", flush=True)

    save_dir            = os.path.join(video_dir, 'video_general_obj_det_finished')
    finished_dir        = os.path.join(video_dir, 'ignore')
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(finished_dir, exist_ok=True)

    decode_dir = os.path.join(video_dir, 'Videos_crop_decode')
    print(f'Input directory: {decode_dir}')
    video_ls = glob.glob(f'{decode_dir}/*')

    random.seed(0)
    random.shuffle(video_ls)

    if video_names:
        target_video_names = set(video_names)
        videos_to_process = []
        found_videos = set()

        for video_path in video_ls:
            video_name = os.path.basename(video_path)
            if video_name in target_video_names:
                videos_to_process.append(video_path)
                found_videos.add(video_name)

        missing_videos = target_video_names - found_videos
        if missing_videos:
            print(f'WARNING: Videos not found in {decode_dir}: {list(missing_videos)}')

        if videos_to_process:
            print(f'Processing SPECIFIC videos: {len(videos_to_process)} videos')
        else:
            print(f'ERROR: None of the specified videos found')
            exit(1)
    elif single_video_name:
        target_video_path = None
        for video_path in video_ls:
            video_name = os.path.basename(video_path)
            if video_name == single_video_name:
                target_video_path = video_path
                break

        if target_video_path:
            videos_to_process = [target_video_path]
            print(f'Processing SINGLE video: {single_video_name}')
        else:
            print(f'ERROR: Video {single_video_name} not found')
            exit(1)
    elif chunk_idx == -1:
        videos_to_process = video_ls
        print(f'Processing ALL {len(videos_to_process)} videos')
    else:
        video_chunk_ls = chunk_into_n(video_ls, chunk_num)
        videos_to_process = video_chunk_ls[chunk_idx]
        print(f'Processing chunk {chunk_idx}/{chunk_num-1} ({len(videos_to_process)} videos)')

    skipped_videos = []
    skipped_reasons = {}

    total_video_processing_start = time.time()

    # --- LISTA ESTESA DELLE CLASSI OGGETTI (Esempio adattabile a 35+ classi) ---
    ALL_OBJECT_QUERIES = [
        "chair", "table", "cup", "sofa", "bottle", "screen/monitor", "guitar", "camera", "laptop",
        "cellphone", "stool", "dish", "piano", "cake", "electric_fan", "faucet", "sink", "refrigerator",
        "microwave", "bread", "vegetables", "oven", "toilet"
    ]

    PROMPT_CHUNK_SIZE = 12
    PROMPT_CHUNKS = [ALL_OBJECT_QUERIES[i:i + PROMPT_CHUNK_SIZE] for i in range(0, len(ALL_OBJECT_QUERIES), PROMPT_CHUNK_SIZE)]

    for video_idx, video_dir in enumerate(tqdm(videos_to_process)):
        video_start_time = time.time()
        video_name = video_dir.split('/')[-1].split('.')[0]

        print(f'Processing video {video_idx+1}/{len(videos_to_process)}: {video_name}', flush=True)

        out_dir = os.path.join(save_dir, video_name)
        out_path = os.path.join(out_dir, video_name+'.mp4')
        pkl_path = os.path.join(out_dir, video_name+'.json')

        finished_path = check_and_merge_existing_markers(finished_dir, video_name)

        if os.path.exists(out_path) or os.path.exists(finished_path):
            continue

        camera_motion_marker = os.path.join(finished_dir, f'{video_name}_camera_motion_skipped')
        if os.path.exists(camera_motion_marker):
            continue

        img_ls = glob.glob(f'{video_dir}/*.jpg')
        img_ls.sort()

        if len(img_ls) > max_video_frames:
            print(f'Skipping video: Too many frames ({len(img_ls)})')
            os.makedirs(finished_path, exist_ok=True)
            continue

        video_image_cache = preload_video_images(img_ls, max_preload=min(10, len(img_ls)))
        first_img = get_cached_image(img_ls[0], video_image_cache)
        img_height, img_width = first_img.shape[:2]

        camera_motion_start = time.time()
        if enable_camera_motion_detection and vggt_model is not None:
            should_skip, motion_score, skip_reason = should_skip_video_due_to_camera_motion(
                vggt_model, video_dir, motion_threshold=camera_motion_threshold, sample_frames=camera_motion_sample_frames
            )

            if should_skip:
                print(f'Skipping video: Camera motion')
                skipped_videos.append(video_name)
                skipped_reasons[video_name] = {
                    'reason': 'Camera motion detected',
                    'motion_score': motion_score,
                    'threshold': camera_motion_threshold,
                    'detail': skip_reason
                }
                camera_motion_marker = os.path.join(finished_dir, f'{video_name}_camera_motion_skipped')
                os.makedirs(camera_motion_marker, exist_ok=True)
                with open(os.path.join(camera_motion_marker, 'skip_reason.txt'), 'w') as f:
                    f.write(f'Video skipped due to camera motion\nMotion score: {motion_score:.4f}\n')
                continue

        camera_motion_time = time.time() - camera_motion_start
        timing_stats['camera_motion_detection'] += camera_motion_time

        object_analysis_start = time.time()
        sam2_propagation_start = time.time()

        RE_DETECTION_INTERVAL = 30
        video_segments = {}
        global_object_id_to_class = {}
        global_obj_counter = 1000
        all_active_video_obj_ids = set()

        print_step_header("Detecting Multi-Frame Re-Detection Chunked with Multi-Pass VLM and SAM2", 5)

        with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):
            inference_state = predictor.init_state(
                video_path=video_dir,
                offload_video_to_cpu=True,
                offload_state_to_cpu=True,
                async_loading_frames=False
            )

        for start_frame_idx in range(0, len(img_ls), RE_DETECTION_INTERVAL):
            end_frame_idx = min(start_frame_idx + RE_DETECTION_INTERVAL, len(img_ls))
            print(f" -> Elaborazione chunk temporale: frame {start_frame_idx} fino a {end_frame_idx-1}")

            current_img = get_cached_image(img_ls[start_frame_idx], video_image_cache)
            chunk_objects = []

            # --- 1. HUMAN DETECTION (Detectron2) ---
            with torch.cuda.amp.autocast(enabled=False):
                d2_out = detector(current_img)

            det_instances = d2_out['instances']
            valid_human_idx = (det_instances.pred_classes == 0) & (det_instances.scores > 0.70)
            human_bboxes = det_instances.pred_boxes.tensor[valid_human_idx].cpu().numpy().astype(np.float32)
            human_scores = det_instances.scores[valid_human_idx].cpu().numpy().astype(np.float32)

            del d2_out, det_instances
            torch.cuda.empty_cache()

            for bbox, score in zip(human_bboxes, human_scores):
                if get_bbox_area(bbox) > 50:
                    chunk_objects.append({
                        'bbox': bbox.tolist(),
                        'class_name': 'person',
                        'confidence': float(score)
                    })

            # --- 2. MULTI-PASS OBJECT DETECTION (OWLv2 Chunked) ---
            raw_candidates = []
            for p_chunk in PROMPT_CHUNKS:
              try:
                with torch.cuda.amp.autocast(enabled=True):
                    r_bboxes, r_phrases, r_scores = detect_objects_owlv2(
                        owlv2_model, owlv2_processor, current_img, p_chunk, device,
                        box_thresh=0.30
                    )

              except RuntimeError as e:
                if "out of memory" in str(e):
                  print("OOM intercettato. Svuoto la cache e riprovo...")
                  torch.cuda.empty_cache()
                  with torch.cuda.amp.autocast(enabled=True):
                    r_bboxes, r_phrases, r_scores = detect_objects_owlv2(
                        owlv2_model, owlv2_processor, current_img, p_chunk, device,
                        box_thresh=0.30
                    )
                else:
                  raise e

              for bbox, phrase, score in zip(r_bboxes, r_phrases, r_scores):
                    name = phrase.strip().lower()
                    if get_bbox_area(bbox) > 50:
                        raw_candidates.append({
                            'bbox': bbox,
                            'class_name': name,
                            'confidence': float(score)
                        })

            torch.cuda.empty_cache()

            # --- 3. CONSOLIDAMENTO CROSS-PROMPT (Cross-Prompt NMS) ---
            raw_candidates = sorted(raw_candidates, key=lambda x: x['confidence'], reverse=True)
            consolidated_objects = []
            cross_prompt_iou_threshold = 0.45

            for cand in raw_candidates:
                is_duplicate = False
                for cons in consolidated_objects:
                  if cand['class_name'] == cons['class_name'] and get_iou(cand['bbox'], cons['bbox']) > cross_prompt_iou_threshold:
                    is_duplicate = True
                    break
                if not is_duplicate:
                    consolidated_objects.append(cand)

            # Unione dei target umani e degli oggetti consolidati
            chunk_objects.extend(consolidated_objects)

            # Filtro geometrico standard e deduplicazione finale sul chunk
            chunk_objects = filter_and_deduplicate_objects(
                chunk_objects, confidence_threshold=0.30, iou_threshold=0.45
            )
            chunk_objects = remove_contained_boxes(chunk_objects, containment_threshold=0.80)


            if not chunk_objects:
                print(f"No object in frame {start_frame_idx}. Continue.")
                continue

            if start_frame_idx == 0:
                for obj in chunk_objects:
                    obj_id = global_obj_counter
                    global_obj_counter += 1
                    global_object_id_to_class[obj_id] = obj['class_name']
                    all_active_video_obj_ids.add(obj_id)

                    bbox_tensor = torch.tensor(obj['bbox'], dtype=torch.float32, device='cpu')
                    predictor.add_new_points_or_box(
                        inference_state=inference_state, frame_idx=start_frame_idx,
                        obj_id=obj_id, box=bbox_tensor
                    )
            else:
                prev_frame_idx = start_frame_idx - 1
                prev_tracks = video_segments.get(prev_frame_idx, {})

                if prev_tracks and chunk_objects:
                    track_ids = list(prev_tracks.keys())
                    track_boxes = [prev_tracks[tid] for tid in track_ids]
                    det_boxes = [obj['bbox'] for obj in chunk_objects]

                    cost_matrix = np.ones((len(track_boxes), len(det_boxes)))
                    for i, t_box in enumerate(track_boxes):
                        for j, d_box in enumerate(det_boxes):
                            cost_matrix[i, j] = 1.0 - get_iou(t_box, d_box)

                    row_ind, col_ind = linear_sum_assignment(cost_matrix)
                    matched_det_indices = set()
                    iou_threshold = 0.15

                    for r, c in zip(row_ind, col_ind):
                        if cost_matrix[r, c] <= (1.0 - iou_threshold):
                            obj_id = track_ids[r]
                            matched_det_indices.add(c)
                            global_object_id_to_class[obj_id] = chunk_objects[c]['class_name']

                            bbox_tensor = torch.tensor(chunk_objects[c]['bbox'], dtype=torch.float32, device='cpu')
                            predictor.add_new_points_or_box(
                                inference_state=inference_state, frame_idx=start_frame_idx,
                                obj_id=obj_id, box=bbox_tensor
                            )

                    for j, obj in enumerate(chunk_objects):
                        if j not in matched_det_indices:
                            obj_id = global_obj_counter
                            global_obj_counter += 1
                            global_object_id_to_class[obj_id] = obj['class_name']
                            all_active_video_obj_ids.add(obj_id)

                            bbox_tensor = torch.tensor(obj['bbox'], dtype=torch.float32, device='cpu')
                            predictor.add_new_points_or_box(
                                inference_state=inference_state, frame_idx=start_frame_idx,
                                obj_id=obj_id, box=bbox_tensor
                            )
                else:
                    for obj in chunk_objects:
                        obj_id = global_obj_counter
                        global_obj_counter += 1
                        global_object_id_to_class[obj_id] = obj['class_name']
                        all_active_video_obj_ids.add(obj_id)

                        bbox_tensor = torch.tensor(obj['bbox'], dtype=torch.float32, device='cpu')
                        predictor.add_new_points_or_box(
                            inference_state=inference_state, frame_idx=start_frame_idx,
                            obj_id=obj_id, box=bbox_tensor
                        )

            if inference_state is not None and all_active_video_obj_ids:
                with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16):
                    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
                        inference_state, start_frame_idx=start_frame_idx
                    ):
                        if start_frame_idx <= out_frame_idx < end_frame_idx:
                            if out_frame_idx not in video_segments:
                                video_segments[out_frame_idx] = {}

                            for i, out_obj_id in enumerate(out_obj_ids):
                                if out_obj_id in all_active_video_obj_ids:
                                    mask = (out_mask_logits[i] > 0.0).cpu().numpy()
                                    boxes = sv.mask_to_xyxy(mask)
                                    if len(boxes) > 0:
                                        video_segments[out_frame_idx][out_obj_id] = list(boxes[0])

                        if out_frame_idx >= end_frame_idx:
                            break

        os.makedirs(out_dir, exist_ok=True)
        object_analysis_time = time.time() - object_analysis_start
        timing_stats['object_analysis_detection'] += object_analysis_time
        sam2_propagation_time = time.time() - sam2_propagation_start
        timing_stats['sam2_propagation'] += sam2_propagation_time

        # --- REFORMATTING AND VISUALIZATION ---
        reformatting_start = time.time()
        print_step_header("Reformatting and visualization", 6)

        original_fps = get_original_video_fps(video_name, video_dir.replace('/Videos_crop_decode/' + video_name, ''))
        if original_fps is None:
            frame_count = len(img_ls)
            original_fps = calculate_fps_from_frames(frame_count, estimated_duration=5.0)

        writer = imageio.get_writer(out_path, fps=original_fps, codec='libx264')
        motion = {'detected_objects': {}}
        total_frames_count = len(img_ls)

        for obj_id, class_name in global_object_id_to_class.items():
            obj_key = f"{class_name}_{obj_id}"
            if class_name not in OBJECTS_CLASS_ID:
                OBJECTS_CLASS_ID[class_name] = len(OBJECTS_CLASS_ID) % len(HEX_COLOR_PALETTE)

            motion['detected_objects'][obj_key] = {
                'class_name': class_name,
                'class_id': OBJECTS_CLASS_ID[class_name],
                'track_id': obj_id,
                'bbox': [None] * total_frames_count
            }

        for f_idx in range(total_frames_count):
            if f_idx in video_segments:
                for obj_id, bbox in video_segments[f_idx].items():
                    class_name = global_object_id_to_class.get(obj_id)
                    if class_name and get_bbox_area(bbox) > 50:
                        x1, y1, x2, y2 = bbox
                        bbox_norm = [
                            round(x1 / img_width, 4), round(y1 / img_height, 4),
                            round(x2 / img_width, 4), round(y2 / img_height, 4)
                        ]
                        obj_key = f"{class_name}_{obj_id}"
                        if obj_key in motion['detected_objects']:
                            motion['detected_objects'][obj_key]['bbox'][f_idx] = bbox_norm

        for i_idx, img_path in tqdm(enumerate(img_ls)):
            img_name = img_path.split('/')[-1]
            img_cv2 = get_cached_image(img_path, video_image_cache)
            annotated_frame = img_cv2.copy()
            drawing_objects = []

            for obj_key, obj_data in motion['detected_objects'].items():
                current_bbox_norm = obj_data['bbox'][i_idx]
                if current_bbox_norm is not None:
                    x1_norm, y1_norm, x2_norm, y2_norm = current_bbox_norm
                    x1 = x1_norm * img_width
                    y1 = y1_norm * img_height
                    x2 = x2_norm * img_width
                    y2 = y2_norm * img_height
                    drawing_objects.append((obj_data['class_name'], [x1, y1, x2, y2]))

            for (name, bbox) in drawing_objects:
                x1, y1, x2, y2 = bbox
                if x1 >= x2 or y1 >= y2 or x1 < 0 or y1 < 0 or x2 > img_width or y2 > img_height:
                    continue

                if name not in OBJECTS_CLASS_ID:
                    OBJECTS_CLASS_ID[name] = len(OBJECTS_CLASS_ID) % len(HEX_COLOR_PALETTE)

                class_id = OBJECTS_CLASS_ID[name]
                detections = sv.Detections(
                    xyxy=np.array(bbox)[None, :],
                    class_id=np.array([class_id]),
                )

                box_annotator = sv.BoxAnnotator(color=ColorPalette.from_hex(HEX_COLOR_PALETTE))
                annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
                label_annotator = sv.LabelAnnotator(color=ColorPalette.from_hex(HEX_COLOR_PALETTE), smart_position=True)
                annotated_frame = label_annotator.annotate(annotated_frame, detections=detections, labels=[name])

            writer.append_data(annotated_frame[:, :, ::-1])
            cv2.imwrite(os.path.join(out_dir, img_name), annotated_frame)
            del annotated_frame
            del img_cv2

        writer.close()
        reformatting_time = time.time() - reformatting_start
        timing_stats['reformatting_visualization'] += reformatting_time

        # --- SAVE RESULTS ---
        saving_start = time.time()
        print_step_header("Saving results", 7)

        with open(pkl_path, 'w') as f:
            json.dump(motion, f, indent=4, cls=NpEncoder)

        saving_time = time.time() - saving_start
        timing_stats['saving_results'] += saving_time
        video_total_time = time.time() - video_start_time
        timing_stats['total_video_processing'] += video_total_time

        print(f"Video {video_name} completed in {video_total_time:.1f}s", flush=True)

        if 'video_image_cache' in locals():
            video_image_cache.clear()
            del video_image_cache
        if 'motion' in locals():
            motion.clear()
            del motion
        if 'video_segments' in locals():
            video_segments.clear()
            del video_segments
        if 'inference_state' in locals():
            try:
                predictor.reset_state(inference_state)
            except:
                pass
            del inference_state

        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        import ctypes
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass

    total_script_time = time.time() - script_start_time
    total_videos = len(videos_to_process)
    processed_videos = total_videos - len(skipped_videos)

    print(f'\nPROCESSING COMPLETED:')
    print(f'   • Time: {total_script_time/60:.1f}m, Processed: {processed_videos}/{total_videos}')
    if processed_videos > 0:
        avg_time = timing_stats["total_video_processing"]/processed_videos
        print(f'   • Avg time: {avg_time:.1f}s')


propagate in video:  13%|█▎        | 27/210 [00:43<04:59,  1.64s/it]